In [1]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns
import os
import pickle
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve

In [2]:
X_train_feat = pd.read_csv('D:\\Credit Risk Modelling\\Data\\featured\\train_featured.csv')
X_test_feat = pd.read_csv('D:\\Credit Risk Modelling\\Data\\featured\\test_featured.csv')
y_train = pd.read_csv('D:\\Credit Risk Modelling\\Data\\featured\\y_train.csv')
y_test = pd.read_csv('D:\\Credit Risk Modelling\\Data\\featured\\y_test.csv')

### With Class Imbalance

In [3]:
model = RandomForestClassifier(random_state=42)
model.fit(X_train_feat, y_train)
y_pred = model.predict(X_test_feat)
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))
roc_auc = roc_auc_score(y_test, model.predict_proba(X_test_feat)[:, 1])
print(f'ROC AUC Score: {roc_auc}')

d:\Credit Risk Modelling\venv\Lib\site-packages\sklearn\base.py:1336: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


              precision    recall  f1-score   support

           0       0.97      0.99      0.98     11390
           1       0.84      0.71      0.77      1108

    accuracy                           0.96     12498
   macro avg       0.90      0.85      0.87     12498
weighted avg       0.96      0.96      0.96     12498

[[11238   152]
 [  325   783]]
ROC AUC Score: 0.9842874314982742


In [4]:
from scipy.stats import randint

rf = RandomForestClassifier(random_state=42)

param_dist_imbalanced = {
    'n_estimators': randint(200, 800),
    'max_depth': randint(4, 25),
    'min_samples_split': randint(2, 20),
    'min_samples_leaf': randint(1, 20),
    'max_features': ['sqrt', 'log2', None],
    'bootstrap': [True],
    'class_weight': ['balanced', 'balanced_subsample']  # 🔥 important
}

random_search_rf_imbalanced = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_dist_imbalanced,
    n_iter=50,
    scoring='roc_auc',
    cv=5,
    verbose=2,
    random_state=42,
    n_jobs=-1
)

random_search_rf_imbalanced.fit(X_train_feat, y_train)

best_rf_imbalanced = random_search_rf_imbalanced.best_estimator_

print("Best Parameters:", random_search_rf_imbalanced.best_params_)
print("Best CV AUC:", random_search_rf_imbalanced.best_score_)

Fitting 5 folds for each of 50 candidates, totalling 250 fits


d:\Credit Risk Modelling\venv\Lib\site-packages\sklearn\base.py:1336: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


Best Parameters: {'bootstrap': True, 'class_weight': 'balanced_subsample', 'max_depth': 23, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 503}
Best CV AUC: 0.9848832166841314


In [5]:
y_pred_proba = best_rf_imbalanced.predict_proba(X_test_feat)[:, 1]
test_auc = roc_auc_score(y_test, y_pred_proba)
print(classification_report(y_test, best_rf_imbalanced.predict(X_test_feat)))
print("Test AUC:", test_auc)

              precision    recall  f1-score   support

           0       0.98      0.98      0.98     11390
           1       0.79      0.79      0.79      1108

    accuracy                           0.96     12498
   macro avg       0.89      0.89      0.89     12498
weighted avg       0.96      0.96      0.96     12498

Test AUC: 0.9861102747042025


### With Balanced Class

In [6]:
from imblearn.over_sampling import SMOTE
smote = SMOTE(random_state=42)
X_train_feat_sm, y_train_sm = smote.fit_resample(X_train_feat, y_train)

print("Before SMOTE:", y_train.value_counts())
print("After SMOTE:", y_train_sm.value_counts())

Before SMOTE: default
0          34298
1           3189
Name: count, dtype: int64
After SMOTE: default
0          34298
1          34298
Name: count, dtype: int64


In [7]:
model = RandomForestClassifier(random_state=42)
model.fit(X_train_feat_sm, y_train_sm)
y_pred = model.predict(X_test_feat)
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))
roc_auc = roc_auc_score(y_test, model.predict_proba(X_test_feat)[:, 1])
print(f'ROC AUC Score: {roc_auc}')

d:\Credit Risk Modelling\venv\Lib\site-packages\sklearn\base.py:1336: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


              precision    recall  f1-score   support

           0       0.99      0.97      0.98     11390
           1       0.72      0.86      0.78      1108

    accuracy                           0.96     12498
   macro avg       0.85      0.91      0.88     12498
weighted avg       0.96      0.96      0.96     12498

[[11016   374]
 [  160   948]]
ROC AUC Score: 0.983217631845022


In [8]:
param_dist_balanced = {
    'n_estimators': randint(200, 800),
    'max_depth': randint(4, 25),
    'min_samples_split': randint(2, 20),
    'min_samples_leaf': randint(1, 20),
    'max_features': ['sqrt', 'log2', None],
    'bootstrap': [True],
    'class_weight': [None]   # 🔥 no weighting
}

random_search_rf_balanced = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_dist_balanced,
    n_iter=50,
    scoring='roc_auc',
    cv=5,
    verbose=2,
    random_state=42,
    n_jobs=-1
)

random_search_rf_balanced.fit(X_train_feat_sm, y_train_sm)

best_rf_balanced = random_search_rf_balanced.best_estimator_

print("Best Parameters:", random_search_rf_balanced.best_params_)
print("Best CV AUC:", random_search_rf_balanced.best_score_)

Fitting 5 folds for each of 50 candidates, totalling 250 fits


d:\Credit Risk Modelling\venv\Lib\site-packages\sklearn\base.py:1336: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


Best Parameters: {'bootstrap': True, 'class_weight': None, 'max_depth': 18, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 8, 'n_estimators': 720}
Best CV AUC: 0.997314055190629


In [9]:
y_pred_proba = best_rf_balanced.predict_proba(X_test_feat)[:, 1]
test_auc = roc_auc_score(y_test, y_pred_proba)
print(classification_report(y_test, best_rf_balanced.predict(X_test_feat)))
print("Test AUC:", test_auc)

              precision    recall  f1-score   support

           0       0.99      0.96      0.98     11390
           1       0.70      0.88      0.78      1108

    accuracy                           0.96     12498
   macro avg       0.84      0.92      0.88     12498
weighted avg       0.96      0.96      0.96     12498

Test AUC: 0.9843342218615988
